# Nail Retouch SD1.5 LoRA v3 - Strict Plus

This notebook prepares your processed manicure dataset and trains a first-pass **Stable Diffusion 1.5 LoRA without masks**.

Important: this v3 notebook trains on the **retouched target images plus structured captions** from `metadata.jsonl`. It uses the expanded strict-plus subset, and is meant to be used later with **img2img inference** for photo retouching.

## 1. Upload project

Clone the repository to Colab, then mount Google Drive and copy `processed_strict_plus` into `/content/nail-retouch-assistant/dataset`.

Expected files:
- `dataset/processed_strict_plus/train/metadata.jsonl`
- `dataset/processed_strict_plus/val/metadata.jsonl`
- `colab/training_config.yaml`
- `colab/requirements.txt`

In [1]:
from google.colab import drive
from pathlib import Path
import shutil
import subprocess

drive.mount('/content/drive')

PROJECT_ROOT = '/content/nail-retouch-assistant'
CONFIG_PATH = f'{PROJECT_ROOT}/colab/training_config.yaml'
REQUIREMENTS_PATH = f'{PROJECT_ROOT}/colab/requirements.txt'
DRIVE_DATASET_DIR = Path('/content/drive/MyDrive/processed_strict_plus')
LOCAL_DATASET_DIR = Path(f'{PROJECT_ROOT}/dataset/processed_strict_plus')

if not DRIVE_DATASET_DIR.exists():
    raise FileNotFoundError(f'Missing dataset in Drive: {DRIVE_DATASET_DIR}')

train_meta = DRIVE_DATASET_DIR / 'train' / 'metadata.jsonl'
val_meta = DRIVE_DATASET_DIR / 'val' / 'metadata.jsonl'
if not train_meta.exists() or not val_meta.exists():
    raise FileNotFoundError(f'Dataset looks incomplete under {DRIVE_DATASET_DIR}')

shutil.rmtree(LOCAL_DATASET_DIR, ignore_errors=True)
LOCAL_DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(
    f"cp -a '{DRIVE_DATASET_DIR}' '{LOCAL_DATASET_DIR}'",
    shell=True,
    check=True,
    executable='/bin/bash',
)
print(f'Copied dataset to {LOCAL_DATASET_DIR}')

subprocess.run(f"python -m pip install -q -r {REQUIREMENTS_PATH}", shell=True, check=True)
if not Path('/content/kohya_ss').exists():
    subprocess.run(
        "git clone --recurse-submodules https://github.com/bmaltais/kohya_ss.git /content/kohya_ss",
        shell=True,
        check=True,
    )
else:
    subprocess.run(
        "cd /content/kohya_ss && git submodule update --init --recursive",
        shell=True,
        check=True,
        executable='/bin/bash',
    )

req = Path('/content/kohya_ss/requirements.txt')
lines = []
for line in req.read_text(encoding='utf-8').splitlines():
    if line.strip().startswith('-e ./sd-scripts'):
        continue
    lines.append(line)
Path('/content/kohya_ss/requirements.colab.txt').write_text('\n'.join(lines) + '\n', encoding='utf-8')

subprocess.run("python -m pip install -q -r /content/kohya_ss/requirements.colab.txt", shell=True, check=True)
subprocess.run("python -m pip install -q xformers", shell=True, check=True)
print('setup complete')

Mounted at /content/drive
Copied dataset to /content/nail-retouch-assistant/dataset/processed_strict_keep
setup complete


In [2]:
  from pathlib import Path
import json
import yaml
import torch

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

project_root = Path(cfg['project_root'])
processed_dir = project_root / cfg['processed_dir']
prepared_dir = Path(cfg['prepared_data_dir'])
output_dir = Path(cfg['output_dir'])
logging_dir = Path(cfg['logging_dir'])

prepared_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
logging_dir.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError('GPU is not available. Switch Colab runtime to T4/L4/A100 before training.')
print('GPU:', torch.cuda.get_device_name(0))

cfg

GPU: Tesla T4


{'project_name': 'nail-retouch-sd15-v1',
 'base_model': 'runwayml/stable-diffusion-v1-5',
 'project_root': '/content/nail-retouch-assistant',
 'processed_dir': 'dataset/processed_strict_keep',
 'train_metadata': 'train/metadata.jsonl',
 'val_metadata': 'val/metadata.jsonl',
 'colab_workdir': '/content/workdir',
 'kohya_repo': '/content/kohya_ss',
 'prepared_data_dir': '/content/workdir/train_data',
 'output_dir': '/content/workdir/outputs',
 'logging_dir': '/content/workdir/logs',
 'output_name': 'nail_retouch_sd15_v1',
 'sample_prompts': ['professional manicure photo retouch, clean cuticles, clean sidewalls, refined nail shape, glossy nail surface, natural skin texture, realistic hand photo, square, french, nude',
  'professional manicure photo retouch, clean cuticles, clean sidewalls, refined nail shape, glossy nail surface, natural skin texture, realistic hand photo, almond, glitter, nude',
  'professional manicure photo retouch, clean cuticles, clean sidewalls, refined nail shape, 

In [3]:
from pathlib import Path
import json
import shutil

def load_jsonl(path: Path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

train_rows = load_jsonl(processed_dir / cfg['train_metadata'])
shutil.rmtree(prepared_dir, ignore_errors=True)
prepared_dir.mkdir(parents=True, exist_ok=True)
instance_dir = prepared_dir / '10_manicure'
instance_dir.mkdir(parents=True, exist_ok=True)

def export_training_example(row):
    target_src = processed_dir / 'train' / row['target']
    image_name = f"{row['id']}.png"
    image_dst = instance_dir / image_name
    txt_dst = instance_dir / f"{row['id']}.txt"
    shutil.copy2(target_src, image_dst)
    txt_dst.write_text(row['prompt'], encoding='utf-8')

for row in train_rows:
    export_training_example(row)

val_prompt_file = output_dir / 'sample_prompts.txt'
val_prompt_file.write_text('\n'.join(cfg.get('sample_prompts', [])), encoding='utf-8')

print('train examples:', len(train_rows))
print('prepared dir:', instance_dir)
print('sample prompt file:', val_prompt_file)
!find /content/workdir/train_data -maxdepth 2 -type f | head -20

train examples: 21
prepared dir: /content/workdir/train_data/10_manicure
sample prompt file: /content/workdir/outputs/sample_prompts.txt
/content/workdir/train_data/10_manicure/pair_0069.png
/content/workdir/train_data/10_manicure/pair_0022.txt
/content/workdir/train_data/10_manicure/pair_0074.txt
/content/workdir/train_data/10_manicure/pair_0038.txt
/content/workdir/train_data/10_manicure/pair_0054.png
/content/workdir/train_data/10_manicure/pair_0078.txt
/content/workdir/train_data/10_manicure/pair_0070.png
/content/workdir/train_data/10_manicure/pair_0035.txt
/content/workdir/train_data/10_manicure/pair_0028.txt
/content/workdir/train_data/10_manicure/pair_0066.png
/content/workdir/train_data/10_manicure/pair_0069.txt
/content/workdir/train_data/10_manicure/pair_0038.png
/content/workdir/train_data/10_manicure/pair_0063.txt
/content/workdir/train_data/10_manicure/pair_0035.png
/content/workdir/train_data/10_manicure/pair_0071.png
/content/workdir/train_data/10_manicure/pair_0022.png

In [4]:
import textwrap

train_cfg = cfg['training']
cmd = f"""
cd {cfg['kohya_repo']} && \
accelerate launch --num_cpu_threads_per_process 1 ./sd-scripts/train_network.py \
  --pretrained_model_name_or_path={cfg['base_model']} \
  --train_data_dir={prepared_dir} \
  --output_dir={output_dir} \
  --logging_dir={logging_dir} \
  --output_name={cfg['output_name']} \
  --resolution={train_cfg['resolution']} \
  --save_model_as=safetensors \
  --network_module=networks.lora \
  --network_dim={train_cfg['network_dim']} \
  --network_alpha={train_cfg['network_alpha']} \
  --train_batch_size={train_cfg['train_batch_size']} \
  --gradient_accumulation_steps={train_cfg['gradient_accumulation_steps']} \
  --max_train_epochs={train_cfg['max_train_epochs']} \
  --learning_rate={train_cfg['learning_rate']} \
  --unet_lr={train_cfg['unet_lr']} \
  --text_encoder_lr={train_cfg['text_encoder_lr']} \
  --optimizer_type={train_cfg['optimizer_type']} \
  --lr_scheduler={train_cfg['lr_scheduler']} \
  --save_every_n_epochs={train_cfg['save_every_n_epochs']} \
  --mixed_precision={train_cfg['mixed_precision']} \
  --caption_extension=.txt \
  --cache_latents \
  --cache_latents_to_disk \
  --max_data_loader_n_workers={train_cfg['max_data_loader_n_workers']} \
  --bucket_reso_steps={train_cfg['bucket_reso_steps']} \
  --min_bucket_reso={train_cfg['min_bucket_reso']} \
  --max_bucket_reso={train_cfg['max_bucket_reso']} \
  --enable_bucket \
  --xformers
"""

cmd = textwrap.dedent(cmd).strip()
print(cmd)

cd /content/kohya_ss && accelerate launch --num_cpu_threads_per_process 1 ./sd-scripts/train_network.py   --pretrained_model_name_or_path=runwayml/stable-diffusion-v1-5   --train_data_dir=/content/workdir/train_data   --output_dir=/content/workdir/outputs   --logging_dir=/content/workdir/logs   --output_name=nail_retouch_sd15_v1   --resolution=512,512   --save_model_as=safetensors   --network_module=networks.lora   --network_dim=8   --network_alpha=8   --train_batch_size=1   --gradient_accumulation_steps=1   --max_train_epochs=12   --learning_rate=1e-4   --unet_lr=1e-4   --text_encoder_lr=5e-5   --optimizer_type=AdamW8bit   --lr_scheduler=cosine   --save_every_n_epochs=4   --mixed_precision=fp16   --caption_extension=.txt   --cache_latents   --cache_latents_to_disk   --max_data_loader_n_workers=0   --bucket_reso_steps=64   --min_bucket_reso=256   --max_bucket_reso=1024   --enable_bucket   --xformers


In [5]:
import subprocess

process = subprocess.Popen(
    cmd,
    shell=True,
    executable='/bin/bash',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end='')

process.wait()
print('\nRETURN CODE:', process.returncode)
if process.returncode != 0:
    raise RuntimeError(f'Training failed with exit code {process.returncode}')

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-03-23 06:34:53.489332: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774247693.520876    6467 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774247693.531035    6467 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774247693.556658    6467 computation_plac

RuntimeError: Training failed with exit code 1

In [ ]:
from pathlib import Path
import shutil

DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/nail-retouch-outputs')
LOCAL_OUTPUT_DIR = output_dir

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
copied = []
for pattern in ('*.safetensors', '*.ckpt', '*.pt', '*.json', '*.txt'):
    for src in LOCAL_OUTPUT_DIR.glob(pattern):
        dst = DRIVE_OUTPUT_DIR / src.name
        shutil.copy2(src, dst)
        copied.append(dst.name)

print(f'Copied {len(copied)} files to {DRIVE_OUTPUT_DIR}')
for name in copied:
    print(name)


## 2. Expected outputs

Training should write files such as:
- `/content/workdir/outputs/nail_retouch_sd15_v1-000004.safetensors`
- `/content/workdir/outputs/nail_retouch_sd15_v1-000008.safetensors`

Use those LoRA weights later with **SD1.5 img2img** at low denoise strength for manicure photo retouching.